# Ego Gate: IEEE evidence package
Private execution notebook. Runs the existing study and a separately specified online extension. No numerical result is assumed in advance. Export all raw outputs and manifests. The new gate is a fixed heuristic, not a learned meta-controller.


In [ ]:
from pathlib import Path
import sys, subprocess, os, json, hashlib, platform
import torch, numpy, scipy, sklearn
torch.set_num_threads(2)
os.environ["OMP_NUM_THREADS"]="2"
os.environ["MKL_NUM_THREADS"]="2"
base=Path("/kaggle/working/ego_gate")
base.mkdir(exist_ok=True)


In [ ]:
(base/'run_digits.py').write_text('"""Reproducible Split-Digits continual-learning benchmark for Ego Gate.\n\nThe benchmark separates *storage selection* from *replay compute*. Every bounded\nbuffer condition receives the same memory budget and the same number of replay\nexamples per Task-B update. Results are written per seed before aggregation.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport csv\nimport json\nimport math\nimport random\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Callable\n\nimport numpy as np\nimport torch\nfrom scipy.stats import ttest_rel\nfrom sklearn.datasets import load_digits\nfrom sklearn.model_selection import train_test_split\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass MLP(nn.Module):\n    def __init__(self, hidden_size: int = 64) -> None:\n        super().__init__()\n        self.fc1 = nn.Linear(64, hidden_size)\n        self.fc2 = nn.Linear(hidden_size, 10)\n\n    def forward(self, x: torch.Tensor, return_hidden: bool = False):\n        hidden = F.relu(self.fc1(x))\n        logits = self.fc2(hidden)\n        return (logits, hidden) if return_hidden else logits\n\n\n@dataclass(frozen=True)\nclass Config:\n    num_seeds: int = 20\n    data_seed: int = 0\n    hidden_size: int = 64\n    epochs_a: int = 60\n    epochs_b: int = 40\n    learning_rate: float = 0.5\n    batch_size: int = 32\n    replay_batch_size: int = 2\n    buffer_size: int = 40\n    candidate_multiplier: int = 5\n    device: str = "cpu"\n\n\nCONDITIONS = (\n    "none",\n    "random",\n    "stratified_random",\n    "doubt",\n    "curiosity",\n    "egogate",\n    "embedding_kcenter",\n    "egogate_diverse",\n    "full",\n)\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef load_split_digits(data_seed: int):\n    digits = load_digits()\n    x = digits.data.astype(np.float32) / 16.0\n    y = digits.target.astype(np.int64)\n\n    task_a = y <= 4\n    task_b = y >= 5\n    xa_train, xa_test, ya_train, ya_test = train_test_split(\n        x[task_a], y[task_a], test_size=0.2, random_state=data_seed, stratify=y[task_a]\n    )\n    xb_train, xb_test, yb_train, yb_test = train_test_split(\n        x[task_b], y[task_b], test_size=0.2, random_state=data_seed, stratify=y[task_b]\n    )\n    to_x = lambda value: torch.from_numpy(value)\n    to_y = lambda value: torch.from_numpy(value).long()\n    return tuple(\n        fn(value)\n        for value, fn in (\n            (xa_train, to_x),\n            (ya_train, to_y),\n            (xa_test, to_x),\n            (ya_test, to_y),\n            (xb_train, to_x),\n            (yb_train, to_y),\n            (xb_test, to_x),\n            (yb_test, to_y),\n        )\n    )\n\n\ndef iter_batches(\n    x: torch.Tensor,\n    y: torch.Tensor,\n    batch_size: int,\n    generator: torch.Generator,\n):\n    order = torch.randperm(len(x), generator=generator)\n    for start in range(0, len(x), batch_size):\n        idx = order[start : start + batch_size]\n        yield x[idx], y[idx]\n\n\ndef train_task_a(model: MLP, x: torch.Tensor, y: torch.Tensor, cfg: Config, seed: int) -> None:\n    model.train()\n    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.learning_rate)\n    generator = torch.Generator().manual_seed(seed + 10_000)\n    for _ in range(cfg.epochs_a):\n        for xb, yb in iter_batches(x, y, cfg.batch_size, generator):\n            xb, yb = xb.to(cfg.device), yb.to(cfg.device)\n            optimizer.zero_grad(set_to_none=True)\n            F.cross_entropy(model(xb), yb).backward()\n            optimizer.step()\n\n\ndef train_task_b(\n    model: MLP,\n    x_new: torch.Tensor,\n    y_new: torch.Tensor,\n    replay_x: torch.Tensor,\n    replay_y: torch.Tensor,\n    cfg: Config,\n    seed: int,\n) -> None:\n    """Train with fixed replay exposure, independent of buffer population size."""\n    model.train()\n    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.learning_rate)\n    generator = torch.Generator().manual_seed(seed + 20_000)\n    for _ in range(cfg.epochs_b):\n        for xb, yb in iter_batches(x_new, y_new, cfg.batch_size, generator):\n            xb, yb = xb.to(cfg.device), yb.to(cfg.device)\n            optimizer.zero_grad(set_to_none=True)\n            train_x, train_y = xb, yb\n            if len(replay_x):\n                ridx = torch.randint(\n                    len(replay_x), (cfg.replay_batch_size,), generator=generator\n                )\n                rx = replay_x[ridx].to(cfg.device)\n                ry = replay_y[ridx].to(cfg.device)\n                train_x = torch.cat((xb, rx), dim=0)\n                train_y = torch.cat((yb, ry), dim=0)\n            loss = F.cross_entropy(model(train_x), train_y)\n            loss.backward()\n            optimizer.step()\n\n\n@torch.no_grad()\ndef accuracy(model: MLP, x: torch.Tensor, y: torch.Tensor, device: str) -> float:\n    model.eval()\n    prediction = model(x.to(device)).argmax(dim=1).cpu()\n    return float((prediction == y).float().mean())\n\n\n@torch.no_grad()\ndef score_examples(model: MLP, x: torch.Tensor, y: torch.Tensor, device: str):\n    """Compute entropy and the exact squared per-example gradient norm for this MLP.\n\n    For an outer product ab^T, ||ab^T||_F^2 = ||a||^2 ||b||^2. This lets us\n    compute the full parameter-gradient norm without one backward pass per sample.\n    """\n    model.eval()\n    x_device, y_device = x.to(device), y.to(device)\n    logits, hidden = model(x_device, return_hidden=True)\n    probabilities = logits.softmax(dim=1)\n    doubt = -(probabilities * probabilities.clamp_min(1e-12).log()).sum(dim=1)\n\n    output_error = probabilities.clone()\n    output_error[torch.arange(len(y_device), device=device), y_device] -= 1.0\n    hidden_error = (output_error @ model.fc2.weight) * (hidden > 0)\n    output_error_sq = output_error.square().sum(dim=1)\n    hidden_error_sq = hidden_error.square().sum(dim=1)\n    curiosity = output_error_sq * (hidden.square().sum(dim=1) + 1.0)\n    curiosity += hidden_error_sq * (x_device.square().sum(dim=1) + 1.0)\n\n    def zscore(values: torch.Tensor) -> torch.Tensor:\n        return (values - values.mean()) / values.std(unbiased=False).clamp_min(1e-12)\n\n    egogate = zscore(doubt) + zscore(curiosity)\n    return {\n        "doubt": doubt.cpu().numpy(),\n        "curiosity": curiosity.cpu().numpy(),\n        "egogate": egogate.cpu().numpy(),\n        "embedding": hidden.cpu().numpy(),\n    }\n\n\ndef stratified_random_indices(y: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:\n    classes = np.unique(y)\n    base, remainder = divmod(k, len(classes))\n    selected: list[int] = []\n    for position, label in enumerate(classes):\n        count = base + int(position < remainder)\n        candidates = np.flatnonzero(y == label)\n        selected.extend(rng.choice(candidates, size=count, replace=False).tolist())\n    return np.asarray(selected, dtype=np.int64)\n\n\ndef kcenter_indices(features: np.ndarray, k: int, candidates: np.ndarray | None = None) -> np.ndarray:\n    pool = np.arange(len(features)) if candidates is None else np.asarray(candidates)\n    if k >= len(pool):\n        return pool.copy()\n    values = features[pool].astype(np.float64, copy=True)\n    scale = values.std(axis=0, keepdims=True)\n    values = (values - values.mean(axis=0, keepdims=True)) / np.maximum(scale, 1e-8)\n    first = int(np.square(values).sum(axis=1).argmax())\n    chosen = [first]\n    min_distance = np.square(values - values[first]).sum(axis=1)\n    min_distance[first] = -1.0\n    for _ in range(1, k):\n        nxt = int(min_distance.argmax())\n        chosen.append(nxt)\n        distance = np.square(values - values[nxt]).sum(axis=1)\n        min_distance = np.minimum(min_distance, distance)\n        min_distance[chosen] = -1.0\n    return pool[np.asarray(chosen)]\n\n\ndef select_indices(\n    condition: str,\n    y: np.ndarray,\n    scores: dict[str, np.ndarray],\n    cfg: Config,\n    seed: int,\n) -> np.ndarray:\n    n, k = len(y), min(cfg.buffer_size, len(y))\n    rng = np.random.default_rng(seed + 30_000)\n    if condition == "none":\n        return np.empty(0, dtype=np.int64)\n    if condition == "full":\n        return np.arange(n)\n    if condition == "random":\n        return rng.choice(n, size=k, replace=False)\n    if condition == "stratified_random":\n        return stratified_random_indices(y, k, rng)\n    if condition in {"doubt", "curiosity", "egogate"}:\n        return np.argsort(scores[condition])[-k:][::-1].copy()\n    if condition == "embedding_kcenter":\n        return kcenter_indices(scores["embedding"], k)\n    if condition == "egogate_diverse":\n        candidate_count = min(n, cfg.candidate_multiplier * k)\n        candidates = np.argsort(scores["egogate"])[-candidate_count:]\n        return kcenter_indices(scores["embedding"], k, candidates)\n    raise ValueError(f"Unknown condition: {condition}")\n\n\ndef buffer_diagnostics(\n    seed: int,\n    condition: str,\n    indices: np.ndarray,\n    labels: np.ndarray,\n    scores: dict[str, np.ndarray],\n) -> dict[str, float | int | str]:\n    row: dict[str, float | int | str] = {\n        "seed": seed,\n        "condition": condition,\n        "buffer_size": len(indices),\n        "label_coverage": len(np.unique(labels[indices])) if len(indices) else 0,\n    }\n    for label in range(10):\n        row[f"class_{label}"] = int(np.sum(labels[indices] == label)) if len(indices) else 0\n    for name in ("doubt", "curiosity", "egogate"):\n        row[f"mean_{name}"] = float(np.mean(scores[name][indices])) if len(indices) else math.nan\n    return row\n\n\ndef write_csv(path: Path, rows: list[dict]) -> None:\n    if not rows:\n        return\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef summarize_existing(output_dir: Path) -> None:\n    path = output_dir / "per_seed.csv"\n    with path.open(newline="", encoding="utf-8") as handle:\n        rows = []\n        for raw in csv.DictReader(handle):\n            rows.append(\n                {\n                    "seed": int(raw["seed"]),\n                    "condition": raw["condition"],\n                    "task_a_pre": float(raw["task_a_pre"]),\n                    "task_a_post": float(raw["task_a_post"]),\n                    "forgetting": float(raw["forgetting"]),\n                    "task_b_post": float(raw["task_b_post"]),\n                }\n            )\n    conditions = tuple(dict.fromkeys(row["condition"] for row in rows))\n    summary, comparisons = aggregate(rows, conditions)\n    write_csv(output_dir / "summary.csv", summary)\n    write_csv(output_dir / "paired_comparisons.csv", comparisons)\n\n\ndef aggregate(rows: list[dict], conditions: tuple[str, ...]):\n    summary: list[dict] = []\n    for condition in conditions:\n        subset = [row for row in rows if row["condition"] == condition]\n        record: dict[str, float | int | str] = {"condition": condition, "n": len(subset)}\n        for metric in ("task_a_pre", "task_a_post", "forgetting", "task_b_post"):\n            values = np.asarray([row[metric] for row in subset], dtype=float)\n            record[f"{metric}_mean"] = float(values.mean())\n            record[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else math.nan\n        summary.append(record)\n\n    comparisons: list[dict] = []\n    requested_pairs = []\n    if "random" in conditions:\n        requested_pairs.extend(("random", condition) for condition in conditions if condition != "random")\n    requested_pairs.extend(\n        pair\n        for pair in (\n            ("egogate", "doubt"),\n            ("egogate", "curiosity"),\n            ("egogate", "egogate_diverse"),\n        )\n        if pair[0] in conditions and pair[1] in conditions\n    )\n    for baseline_name, condition in requested_pairs:\n        baseline = {\n            row["seed"]: row["forgetting"]\n            for row in rows\n            if row["condition"] == baseline_name\n        }\n        candidate = {\n            row["seed"]: row["forgetting"]\n            for row in rows\n            if row["condition"] == condition\n        }\n        seeds = sorted(set(baseline) & set(candidate))\n        baseline_values = np.asarray([baseline[seed] for seed in seeds])\n        condition_values = np.asarray([candidate[seed] for seed in seeds])\n        improvement = baseline_values - condition_values\n        if len(seeds) > 1:\n            test = ttest_rel(baseline_values, condition_values)\n            std_delta = improvement.std(ddof=1)\n            cohen_dz = improvement.mean() / std_delta if std_delta > 0 else math.inf\n        else:\n            test = type("Test", (), {"statistic": math.nan, "pvalue": math.nan})()\n            cohen_dz = math.nan\n        comparisons.append(\n            {\n                "baseline": baseline_name,\n                "condition": condition,\n                "n": len(seeds),\n                "mean_forgetting_reduction": float(improvement.mean()),\n                "relative_reduction": float(improvement.mean() / baseline_values.mean())\n                if baseline_values.mean() != 0\n                else math.nan,\n                "paired_t": float(test.statistic),\n                "p_value": float(test.pvalue),\n                "cohen_dz": float(cohen_dz),\n            }\n        )\n    # Holm correction controls family-wise error across the declared comparisons.\n    order = sorted(range(len(comparisons)), key=lambda index: comparisons[index]["p_value"])\n    running_max = 0.0\n    for rank, index in enumerate(order):\n        adjusted = (len(order) - rank) * comparisons[index]["p_value"]\n        running_max = max(running_max, adjusted)\n        comparisons[index]["holm_p_value"] = min(1.0, running_max)\n    return summary, comparisons\n\n\ndef run(cfg: Config, output_dir: Path, conditions: tuple[str, ...]) -> None:\n    data = load_split_digits(cfg.data_seed)\n    xa_train, ya_train, xa_test, ya_test, xb_train, yb_train, xb_test, yb_test = data\n    rows: list[dict] = []\n    diagnostics: list[dict] = []\n\n    for seed in range(cfg.num_seeds):\n        seed_everything(seed)\n        base_model = MLP(cfg.hidden_size).to(cfg.device)\n        train_task_a(base_model, xa_train, ya_train, cfg, seed)\n        task_a_pre = accuracy(base_model, xa_test, ya_test, cfg.device)\n        scores = score_examples(base_model, xa_train, ya_train, cfg.device)\n        labels = ya_train.numpy()\n\n        for condition in conditions:\n            indices = select_indices(condition, labels, scores, cfg, seed)\n            diagnostics.append(\n                buffer_diagnostics(seed, condition, indices, labels, scores)\n            )\n            model = copy.deepcopy(base_model)\n            train_task_b(\n                model,\n                xb_train,\n                yb_train,\n                xa_train[indices],\n                ya_train[indices],\n                cfg,\n                seed,\n            )\n            task_a_post = accuracy(model, xa_test, ya_test, cfg.device)\n            task_b_post = accuracy(model, xb_test, yb_test, cfg.device)\n            rows.append(\n                {\n                    "seed": seed,\n                    "condition": condition,\n                    "task_a_pre": task_a_pre,\n                    "task_a_post": task_a_post,\n                    "forgetting": task_a_pre - task_a_post,\n                    "task_b_post": task_b_post,\n                }\n            )\n            print(\n                f"seed={seed:02d} condition={condition:20s} "\n                f"A_pre={task_a_pre:.3f} A_post={task_a_post:.3f} "\n                f"forget={task_a_pre - task_a_post:.3f} B={task_b_post:.3f}",\n                flush=True,\n            )\n\n        write_csv(output_dir / "per_seed.csv", rows)\n        write_csv(output_dir / "buffer_diagnostics.csv", diagnostics)\n\n    summary, comparisons = aggregate(rows, conditions)\n    write_csv(output_dir / "summary.csv", summary)\n    write_csv(output_dir / "paired_comparisons.csv", comparisons)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    (output_dir / "config.json").write_text(\n        json.dumps({**asdict(cfg), "conditions": conditions}, indent=2), encoding="utf-8"\n    )\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output-dir", type=Path, default=Path("results/digits"))\n    parser.add_argument("--num-seeds", type=int, default=20)\n    parser.add_argument("--epochs-a", type=int, default=60)\n    parser.add_argument("--epochs-b", type=int, default=40)\n    parser.add_argument("--buffer-size", type=int, default=40)\n    parser.add_argument("--replay-batch-size", type=int, default=2)\n    parser.add_argument("--device", choices=("cpu", "cuda"), default="cpu")\n    parser.add_argument(\n        "--summarize-only",\n        action="store_true",\n        help="Regenerate aggregate CSVs from an existing per_seed.csv without training.",\n    )\n    parser.add_argument("--conditions", nargs="+", choices=CONDITIONS, default=list(CONDITIONS))\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    if args.summarize_only:\n        summarize_existing(args.output_dir)\n        return\n    if args.device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA was requested but is not available")\n    cfg = Config(\n        num_seeds=args.num_seeds,\n        epochs_a=args.epochs_a,\n        epochs_b=args.epochs_b,\n        buffer_size=args.buffer_size,\n        replay_batch_size=args.replay_batch_size,\n        device=args.device,\n    )\n    run(cfg, args.output_dir, tuple(args.conditions))\n\n\nif __name__ == "__main__":\n    main()\n',encoding="utf-8")


In [ ]:
(base/'stream_gate.py').write_text('"""Single-pass supervised admission/consolidation study; no task IDs enter the gate.\n\nTest boundaries are evaluator-only. Run on Kaggle through the generated notebook.\nScores are computed before each block\'s updates. Comparisons share initialization,\nstream, corruption, batch size and optimization-update counts, not total compute.\n"""\nfrom __future__ import annotations\nimport argparse\nimport copy\nimport csv\nimport hashlib\nimport json\nimport platform\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.datasets import load_digits\nfrom sklearn.model_selection import train_test_split\n\n\n@dataclass\nclass Config:\n    dataset: str = "digits"\n    seed: int = 0\n    memory: int = 200\n    batch: int = 32\n    period: int = 8\n    lr: float = 0.05\n    hidden: int = 128\n    noise: float = 0.0\n    epsilon: float = 0.1\n    device: str = "cpu"\n\n\nPOLICIES = ("reservoir", "balanced", "doubt", "curiosity", "egogate", "egogate_audit")\n\n\nclass Net(nn.Module):\n    def __init__(self, dim, hidden):\n        super().__init__()\n        self.fc1 = nn.Linear(dim, hidden)\n        self.fc2 = nn.Linear(hidden, 10)\n\n    def forward(self, x):\n        return self.fc2(F.relu(self.fc1(x)))\n\n\n@torch.no_grad()\ndef scores(model, x, y, mode):\n    h = F.relu(model.fc1(x))\n    p = model.fc2(h).softmax(1)\n    entropy = -(p * p.clamp_min(1e-12).log()).sum(1) / np.log(10)\n    if mode == "doubt":\n        return entropy\n    err = p.clone()\n    err[torch.arange(len(y), device=y.device), y] -= 1\n    back = (err @ model.fc2.weight) * (h > 0)\n    grad2 = err.square().sum(1) * (h.square().sum(1) + 1)\n    grad2 += back.square().sum(1) * (x.square().sum(1) + 1)\n    # Bounded monotone transform avoids full-stream normalization or future data.\n    influence = grad2 / (1 + grad2)\n    return influence if mode == "curiosity" else (entropy + influence) / 2\n\n\nclass Memory:\n    def __init__(self, capacity, policy, seed, epsilon=0.1):\n        self.capacity, self.policy, self.epsilon = capacity, policy, epsilon\n        self.rng = np.random.default_rng(seed)\n        self.items = []\n        self.seen = 0\n        self.admitted = 0\n        self.score_seconds = 0.0\n        self.scored_examples = 0\n\n    def add(self, x, y, corrupt, model):\n        self.seen += 1\n        item = (x.detach().cpu().clone(), int(y), bool(corrupt))\n        if len(self.items) < self.capacity:\n            self.items.append(item)\n            self.admitted += 1\n            return True\n        if self.policy == "reservoir":\n            j = int(self.rng.integers(self.seen))\n            if j < self.capacity:\n                self.items[j] = item\n                self.admitted += 1\n                return True\n            return False\n        if self.policy == "balanced":\n            # Online class-balanced random replacement, using observed labels only.\n            labels = np.array([v[1] for v in self.items])\n            counts = np.bincount(labels, minlength=10)\n            largest = np.flatnonzero(counts == counts.max())\n            victim_class = int(self.rng.choice(largest))\n            if counts[int(y)] <= counts[victim_class]:\n                candidates = np.flatnonzero(labels == victim_class)\n                self.items[int(self.rng.choice(candidates))] = item\n                self.admitted += 1\n                return True\n            return False\n        # Audit bypass keeps some low-score evidence accessible. This is exploratory,\n        # not a calibrated poisoning defense or an adaptive/meta-learned controller.\n        j = int(self.rng.integers(self.capacity))\n        accept = self.policy == "egogate_audit" and self.rng.random() < self.epsilon\n        if not accept:\n            device = next(model.parameters()).device\n            pair_x = torch.stack((item[0], self.items[j][0])).to(device)\n            pair_y = torch.tensor((item[1], self.items[j][1]), device=device)\n            if device.type == "cuda":\n                torch.cuda.synchronize()\n            start = time.perf_counter()\n            s = scores(model, pair_x, pair_y, self.policy)\n            accept = bool(s[0] > s[1])\n            if device.type == "cuda":\n                torch.cuda.synchronize()\n            self.score_seconds += time.perf_counter() - start\n            self.scored_examples += 2\n        if accept:\n            self.items[j] = item\n            self.admitted += 1\n        return accept\n\n    def sample(self, size, rng, device):\n        idx = rng.integers(len(self.items), size=size)\n        return (torch.stack([self.items[i][0] for i in idx]).to(device),\n                torch.tensor([self.items[i][1] for i in idx], device=device))\n\n\ndef dataset(cfg, root):\n    if cfg.dataset == "digits":\n        d = load_digits()\n        x, y = d.data.astype("float32") / 16, d.target\n        a, b = train_test_split(np.arange(len(y)), test_size=.2,\n                                stratify=y, random_state=cfg.seed)\n        return x[a], y[a], x[b], y[b]\n    from torchvision.datasets import FashionMNIST\n    train = FashionMNIST(root, train=True, download=True)\n    test = FashionMNIST(root, train=False, download=True)\n    return (train.data.numpy().reshape(-1, 784).astype("float32") / 255,\n            train.targets.numpy(), test.data.numpy().reshape(-1, 784).astype("float32") / 255,\n            test.targets.numpy())\n\n\ndef run_one(cfg, policy, schedule, root, admission=False):\n    torch.manual_seed(cfg.seed)\n    np.random.seed(cfg.seed)\n    x, y, tx, ty = dataset(cfg, root)\n    stream_rng = np.random.default_rng(cfg.seed + 1000)\n    classes = stream_rng.permutation(10).reshape(5, 2)\n    stream = np.concatenate([stream_rng.permutation(np.flatnonzero(np.isin(y, c))) for c in classes])\n    ends = np.cumsum([np.isin(y, c).sum() for c in classes])\n    clean = y[stream].copy()\n    noisy = clean.copy()\n    corruption = stream_rng.random(len(noisy)) < cfg.noise\n    noisy[corruption] = (noisy[corruption] + stream_rng.integers(1, 10, corruption.sum())) % 10\n    x = torch.tensor(x[stream])\n    noisy = torch.tensor(noisy, dtype=torch.long)\n    model = Net(x.shape[1], cfg.hidden).to(cfg.device)\n    opt = torch.optim.SGD(model.parameters(), lr=cfg.lr)\n    memory = Memory(cfg.memory, policy, cfg.seed + 2000, cfg.epsilon)\n    replay_rng = np.random.default_rng(cfg.seed + 3000)\n    order_rng = np.random.default_rng(cfg.seed + 4000)\n    pending, trace = [], []\n    updates, exposures, peak_pending = 0, 0, 0\n    period = 1 if schedule == "immediate" else cfg.period\n    start_time = time.perf_counter()\n\n    def consolidate():\n        nonlocal updates, exposures\n        if not pending:\n            return\n        # Retain the same number of updates and new examples across schedules.\n        for index in order_rng.permutation(len(pending)):\n            bx, by, original_count = pending[index]\n            rx, ry = memory.sample(cfg.batch + original_count - len(by), replay_rng, cfg.device)\n            allx = torch.cat((bx.to(cfg.device), rx))\n            ally = torch.cat((by.to(cfg.device), ry))\n            opt.zero_grad(set_to_none=True)\n            F.cross_entropy(model(allx), ally).backward()\n            opt.step()\n            updates += 1\n            exposures += len(ally)\n        pending.clear()\n\n    # Only the evaluator uses ends; minibatches continue across task transitions.\n    evaluated = set()\n    for lo in range(0, len(x), cfg.batch):\n        hi = min(lo + cfg.batch, len(x))\n        admitted = [memory.add(x[i], noisy[i], corruption[i], model) for i in range(lo, hi)]\n        mask = torch.tensor(admitted if admission else [True]*(hi-lo), dtype=torch.bool)\n        pending.append((x[lo:hi][mask], noisy[lo:hi][mask], hi-lo))\n        peak_pending = max(peak_pending, sum(len(v[0]) for v in pending))\n        if len(pending) >= period or hi == len(x):\n            consolidate()\n        for task, end in enumerate(ends):\n            if hi >= end and task not in evaluated:\n                evaluated.add(task)\n                with torch.no_grad():\n                    pred = torch.cat([model(torch.tensor(tx[j:j+512], device=cfg.device)).argmax(1).cpu()\n                                      for j in range(0, len(tx), 512)]).numpy()\n                acc = [float((pred[np.isin(ty,c)] == ty[np.isin(ty,c)]).mean()) for c in classes]\n                trace.append({"task":task, "observations":hi, "accuracy":acc,\n                              "pending_examples":sum(len(v[0]) for v in pending)})\n    matrix = np.asarray([v["accuracy"] for v in trace])\n    final = float(matrix[-1].mean())\n    forgetting = float(np.mean([matrix[j:-1,j].max() - matrix[-1,j] for j in range(4)]))\n    row = {**asdict(cfg), "policy":policy, "schedule":schedule, "admission":admission,\n           "final_accuracy":final, "forgetting":forgetting,\n           "mean_seen_accuracy":float(np.mean([matrix[j,:j+1].mean() for j in range(5)])),\n           "updates":updates, "training_exposures":exposures,\n           "admission_fraction":memory.admitted / memory.seen,\n           "buffer_corruption":float(np.mean([v[2] for v in memory.items])),\n           "peak_pending_examples":peak_pending, "buffer_examples":len(memory.items),\n           "score_seconds":memory.score_seconds, "scored_examples":memory.scored_examples,\n           "wall_seconds":time.perf_counter()-start_time,\n           "class_order":classes.tolist(), "trace":trace}\n    return row\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--datasets", nargs="+", default=["digits", "fashion"])\n    p.add_argument("--seeds", type=int, default=5)\n    p.add_argument("--noise", type=float, nargs="+", default=[0., .2])\n    p.add_argument("--output", type=Path, default=Path("/kaggle/working/stream_results"))\n    p.add_argument("--data-root", default="/kaggle/working/data")\n    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    args = p.parse_args()\n    torch.set_num_threads(2)\n    args.output.mkdir(parents=True, exist_ok=True)\n    # Study is fixed before execution: six selectors + schedule ablation.\n    conditions = [(p,"immediate",False) for p in POLICIES] + [\n        ("reservoir","periodic",False),("egogate_audit","periodic",False),\n        ("egogate_audit","immediate",True),("egogate_audit","periodic",True)]\n    manifest = {"python":platform.python_version(), "torch":torch.__version__,\n                "numpy":np.__version__, "device":args.device,\n                "gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n                "source_sha256":hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n                "arguments":{k:str(v) if isinstance(v,Path) else v for k,v in vars(args).items()},\n                "conditions":conditions}\n    (args.output/"manifest.json").write_text(json.dumps(manifest,indent=2))\n    results = []\n    for name in args.datasets:\n        for noise in args.noise:\n            for seed in range(args.seeds):\n                for policy,schedule,admission in conditions:\n                    cfg = Config(dataset=name, seed=seed, noise=noise, device=args.device)\n                    row = run_one(cfg,policy,schedule,args.data_root,admission)\n                    results.append(row)\n                    (args.output/"raw.json").write_text(json.dumps(results,indent=2))\n                    print(name,noise,seed,policy,schedule,admission,round(row["final_accuracy"],4),flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',encoding="utf-8")


In [ ]:
(base/'analyze_stream.py').write_text('"""Validate completed stream outputs and emit summary plus declared paired tests."""\nimport argparse\nimport csv\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom scipy.stats import t, ttest_rel\n\np=argparse.ArgumentParser()\np.add_argument(\'directory\',type=Path)\na=p.parse_args()\nrows=json.loads((a.directory/\'raw.json\').read_text())\nmanifest=json.loads((a.directory/\'manifest.json\').read_text())\nargs=manifest[\'arguments\']\nexpected=len(args[\'datasets\'])*len(args[\'noise\'])*args[\'seeds\']*len(manifest[\'conditions\'])\nkeys=[(r[\'dataset\'],r[\'noise\'],r[\'seed\'],r[\'policy\'],r[\'schedule\'],r[\'admission\']) for r in rows]\nassert len(keys)==len(set(keys))==expected, \'Incomplete or duplicate results; do not report as complete.\'\nfor name in args[\'datasets\']:\n    for noise in args[\'noise\']:\n        for seed in range(args[\'seeds\']):\n            group=[r for r in rows if (r[\'dataset\'],r[\'noise\'],r[\'seed\'])==(name,noise,seed)]\n            assert len({r[\'updates\'] for r in group})==1\n            assert len({r[\'training_exposures\'] for r in group})==1\n            assert all(r[\'buffer_examples\']==r[\'memory\'] for r in group)\n            assert all(len(r[\'trace\'])==5 for r in group)\n\ndef write(name, data):\n    with (a.directory/name).open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(data[0]))\n        writer.writeheader();writer.writerows(data)\n\nsummary=[]\ncomparisons=[]\nfor name in args[\'datasets\']:\n    for noise in args[\'noise\']:\n        for policy,schedule,admission in manifest[\'conditions\']:\n            g=[r for r in rows if (r[\'dataset\'],r[\'noise\'],r[\'policy\'],r[\'schedule\'],r[\'admission\'])==(name,noise,policy,schedule,admission)]\n            d={\'dataset\':name,\'noise\':noise,\'policy\':policy,\'schedule\':schedule,\'admission\':admission,\'n\':len(g)}\n            for metric in [\'final_accuracy\',\'forgetting\',\'mean_seen_accuracy\',\'buffer_corruption\',\'admission_fraction\',\'wall_seconds\',\'score_seconds\',\'peak_pending_examples\',\'updates\',\'training_exposures\']:\n                v=np.array([r[metric] for r in g]);d[metric+\'_mean\']=float(v.mean());d[metric+\'_sd\']=float(v.std(ddof=1))\n            summary.append(d)\n        for label,b,c in [\n            (\'admission\',(\'egogate_audit\',\'immediate\',False),(\'egogate_audit\',\'immediate\',True)),\n            (\'schedule\',(\'egogate_audit\',\'immediate\',True),(\'egogate_audit\',\'periodic\',True))]:\n            def values(condition):\n                g=sorted([r for r in rows if (r[\'dataset\'],r[\'noise\'])==(name,noise) and (r[\'policy\'],r[\'schedule\'],r[\'admission\'])==condition],key=lambda r:r[\'seed\'])\n                return np.array([r[\'final_accuracy\'] for r in g])\n            bv,cv=values(b),values(c);diff=cv-bv\n            sd=diff.std(ddof=1);se=sd/np.sqrt(len(diff));mean=diff.mean()\n            if sd==0:\n                stat,pv=(0.,1.) if mean==0 else (float(np.sign(mean)*np.inf),0.)\n            else:\n                test=ttest_rel(cv,bv);stat,pv=float(test.statistic),float(test.pvalue)\n            ci=(mean-t.ppf(.975,len(diff)-1)*se,mean+t.ppf(.975,len(diff)-1)*se)\n            comparisons.append({\'dataset\':name,\'noise\':noise,\'comparison\':label,\'n\':len(diff),\'accuracy_improvement\':float(mean),\'ci_low\':float(ci[0]),\'ci_high\':float(ci[1]),\'paired_t\':stat,\'p\':pv})\norder=sorted(range(len(comparisons)),key=lambda i:comparisons[i][\'p\'])\nlast=0.\nfor rank,i in enumerate(order):\n    last=max(last,(len(order)-rank)*comparisons[i][\'p\']);comparisons[i][\'holm_p\']=min(last,1.)\nwrite(\'summary.csv\',summary);write(\'primary_comparisons.csv\',comparisons)\nprint(\'Validated\',len(rows),\'runs; wrote descriptive and primary paired analyses.\')\n',encoding="utf-8")


In [ ]:
manifest={'source_hashes': {'experiments/run_digits.py': '1930fdb2956c4f27e4d72c9823dee052e4317c49bc21eb419fa495efc3a5d4fb', 'experiments/stream_gate.py': 'b8bc15dd2ccfadbe53bc3f3cda57df628282579ae3d4fc1e2c2894a92f283df1', 'scripts/analyze_stream.py': '7d951fcaa3bc3b9debc37eef88e27f190d5e59c54b2ef7ac2b5c297adf81749b'}, 'seeds_online': 5}
manifest.update({"python":platform.python_version(),"torch":torch.__version__,"numpy":numpy.__version__,"scipy":scipy.__version__,"sklearn":sklearn.__version__,"kaggle":bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))})
(base/"execution_manifest.json").write_text(json.dumps(manifest,indent=2))


## Study A: independent reproduction of the existing two-task result
The manuscript currently reports archived local results. This run is independent evidence and may differ with platform and package versions.


In [ ]:
subprocess.run([sys.executable,str(base/"run_digits.py"),"--num-seeds","20","--output-dir",str(base/"digits_reproduction"),"--device","cpu"],check=True)


## Study B: single-pass online admission and consolidation
Five two-class phases, randomized class order; boundaries never enter the learner. Clean and 20% label-noise streams; 200 stored examples. Ten conditions separate storage, admission and scheduling. Every method has equal training exposure/update budgets; scoring time and temporary memory are additional reported costs.


In [ ]:
subprocess.run([sys.executable,str(base/"stream_gate.py"),"--seeds",'5',"--output",str(base/"stream_results"),"--device","cpu"],check=True)


In [ ]:
subprocess.run([sys.executable,str(base/"analyze_stream.py"),str(base/"stream_results")],check=True)


## Completion and integrity
A completion marker is written only after both studies finish successfully. Partial raw outputs remain available after a failure.


In [ ]:
(base/"COMPLETE.json").write_text(json.dumps({"complete":True,"source_hashes":manifest["source_hashes"]},indent=2))
print("Both studies complete; download /kaggle/working/ego_gate")
